In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace
from typing import TypedDict
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen3.5-9B",
    task="text-generation"
)

model = ChatHuggingFace(llm=llm)

d:\Langraph\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
class BlogState(TypedDict):
    title : str
    outline : str
    content : str

In [4]:
def create_outline(state: BlogState) -> BlogState:
    # fetch the title from the state
    title = state["title"]
    # create a prompt for the model
    prompt = f"Create an outline for a blog post with the title: {title}"
    # get the outline from the model
    outline = model.invoke(prompt).content
    # update the state with the outline
    state["outline"] = outline
    return state

def create_blog(state: BlogState) -> BlogState:
    # fetch the title and outline from the state
    title = state["title"]
    outline = state["outline"]
    # create a prompt for the model
    prompt = f"Create a detailed blog post with the title: {title} and the following outline: \n{outline}"
    # get the content from the model
    content = model.invoke(prompt).content
    # update the state with the content
    state["content"] = content
    return state

In [7]:
graph = StateGraph(BlogState)

# nodes 
graph.add_node("Create Outline", create_outline)
graph.add_node("Create Blog", create_blog)

# edges
graph.add_edge(START, "Create Outline")
graph.add_edge("Create Outline", "Create Blog")
graph.add_edge("Create Blog", END)

workflow = graph.compile()

In [8]:
initial_state = {"title": "The Future of AI"}
final_state = workflow.invoke(initial_state)
print(final_state)

{'title': 'The Future of AI', 'outline': '', 'content': ''}
